# Lab 10 · Routing: rules, models, and a safe fallback

**Prepared by Mustafa Assaf** · Self-study: Go deeper · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Route a request

A router sends different requests to different handlers. This example uses a fixed rule; later the model selects a route and code handles unknown outputs safely.

In [ ]:
request = "rainy"
if request == "rainy":
    route = "indoor"
else:
    route = "outdoor"
handlers = {"indoor": "Visit the museum", "outdoor": "Walk in the park"}
print("Weather:", request)
print("Route:", route)
print("Handler result:", handlers[route])

**Next:** Now route routine resilience questions to a rule and uncertain cases to review.

## Route resilience questions by uncertainty and risk
**Start with a problem:** Sending every question to a full review can cost time, while an automatic decision can miss an unsafe payment.

**In this lab:** Route clear cases to a simple rule and uncertain or risky cases to review.

**So far:** the human approved retries only for safe effects. **New:** a model router chooses a cheap rule or a careful review queue. ShopEasy's checkout depends on Pricing, Inventory, Payments, and Recommendations.

The rule gives narrow advice for known idempotent or non-idempotent effects. A case with unknown idempotency or a business consequence that needs judgement goes to the council/human queue. Invalid model output also falls back to review. This lab recommends policies; it sends no requests and charges no cards.

**At a glance**

![Lab 10 at a glance: Case (facts + note), then Router (route + reason), then valid?, then Safe rule (0 extra model calls) or Review (uncertain effects), then Compare (vs human routes).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/10.svg)

## Think about the example

A Pricing lookup is a read. A Payments charge can change a customer's balance even if the response times out. A router itself can add model calls before a case reaches review.

**Concepts:** Retry safety and fallback choices; Different costs of routing errors.

Use the scenario above to consider these questions before running the code:

1. Why is the Payments case more dangerous than a failed Pricing read?
2. Why can routing cost as much as sending every case to review?

## The idea in context

Routing chooses which workflow handles a case. A dispatch table and conservative fallback keep selection separate from execution.

A router spends calls too; an uncertain reply should not silently skip review.

| Key term | Meaning |
| --- | --- |
| **Router** | A component that picks the handler for each input. |
| **Dispatch table** | A mapping from each route to the function that handles it. |
| **Fallback** | The safe route taken when the router's answer cannot be used. |
| **Asymmetric errors** | Mistakes whose costs differ, so they are counted and weighed separately. |

## Pattern · Routing

Routing is not a separate entry in the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. It combines two of the guide's ideas: the risk classification of *Human in the Loop* (classify, then send only high-risk cases to the expensive path) and the dispatch idea in *Registry & Adapter*. This simple example uses `if` statements instead of a registry of functions.

| | |
| --- | --- |
| **Context** | Cases differ in how much deliberation they need, and the most careful path is also the most expensive. |
| **Problem** | Sending every case to the council wastes calls; sending every case to a rule skips debates that were needed. |
| **Solution** | Classify each case, dispatch it to the handler that fits, and fall back to the careful path when the classification is unusable. |
| **Use when** | Cases fall into recognizable classes; handlers differ greatly in cost; one kind of wrong route is cheaper than the other. |
| **Trade-off** | Saves calls on routine cases, but the router adds its own calls, and its mistakes stay silent unless they are measured. |
| **Flow** | Case → Classify → Dispatch → Handler |

## Main exercise at a glance

1. Inputs and human routes
2. Prepare the policy and routes
3. Ask the router for each case
4. Apply routes and count mistakes

## Step 1 · Inputs and human routes

These short notes are teaching scenarios created for the exercise, not measured service behaviour. The held-out route labels are used for evaluation, never included in the router's prompt.

In [ ]:
import json

cases = [
    {"name": "Pricing", "idempotent": True, "note": "Transient quote-lookup timeout; bounded retry fits the latency budget."},
    {"name": "Inventory", "idempotent": None, "note": "A reservation timed out; its contract does not state idempotency."},
    {"name": "Payments", "idempotent": False, "note": "The charge outcome is uncertain; reconcile status before another charge."},
    {"name": "Recommendations", "idempotent": True, "note": "Optional and flaky; decide whether fallback is better than more waiting."},
    {"name": "Catalog", "idempotent": True, "note": "Transient product-lookup timeout; bounded retry fits the latency budget."},
]
human_routes = {"Pricing": "rule", "Inventory": "review", "Payments": "review",
                "Recommendations": "review", "Catalog": "rule"}

## Step 2 · Prepare the policy and routes
The rule permits a bounded retry only for known idempotent effects, and it costs no further model calls — the router has already spent the case's one call. The review route adds the case to a queue; its estimate of five calls represents future review work this notebook does not run. Step 4 adds both kinds to the router's own calls.

In [ ]:
POLICY = "Retry only idempotent effects, bounded with exponential backoff and jitter. Never blindly repeat a payment charge."
COST = {"rule": 0, "review": 5}  # estimated future review cost
ROUTES = ["rule", "review"]
review_queue = []
print("Allowed routes:", ROUTES)
print("Payment safety:", POLICY)

## Step 3 · Ask the router for each case
The model chooses `rule` or `review` for every case. The loop sends one prompt at a time. If a reply is not valid JSON or does not name an allowed route, the code chooses `review` instead.

The saved routes are recommendations. The next step applies the rules; the model cannot execute a payment.

In [ ]:
EXAMPLES = ("Transient idempotent lookup, latency budget available -> rule. "
            "Unknown effect idempotency -> review. Uncertain non-idempotent charge -> review. "
            "Optional service needs fallback judgement -> review.")
routes = {}
for case in cases:
    prompt = (f"Policy: {POLICY}\n{EXAMPLES}\nCase: {json.dumps(case)}\n"
              'Reply with JSON only: {"route": "rule" or "review", "reason": "one sentence"}')
    try:
        reply = json.loads(ask(prompt))
    except json.JSONDecodeError:
        reply = {}
    if not isinstance(reply, dict) or reply.get("route") not in ROUTES:
        routes[case["name"]] = ("review", "unknown or malformed route")
    else:
        routes[case["name"]] = (reply["route"], reply.get("reason", ""))
    print(case["name"], "->", routes[case["name"]])

## Step 4 · Apply routes and count mistakes
A simple `if` chooses what happens next. The rule never recommends a blind non-idempotent retry; the review path queues the case. Count missed reviews and unnecessary reviews separately, including the estimated cost of queued work.

In [ ]:
calls, wasted, skipped = len(cases), [], []
for case in cases:
    chosen, reason = routes[case["name"]]
    if chosen == "review":
        review_queue.append(case["name"])
        result = "queued for council/human review"
    elif case["idempotent"] is True:
        result = "bounded retry with backoff and jitter"
    else:
        result = "do not automatically retry"
    calls += COST[chosen]
    if chosen != human_routes[case["name"]]:
        if chosen == "rule":
            skipped.append(case["name"])
        else:
            wasted.append(case["name"])
    print(case["name"], "->", chosen, "|", result, "|", reason)
print("Agreed:", len(cases) - len(wasted) - len(skipped), "/", len(cases))
print("Estimated calls including queued reviews:", calls, "| all-review:", COST["review"] * len(cases))
print("Wasted reviews:", wasted, "| skipped reviews:", skipped)
assert not skipped, "A case requiring judgement skipped review."

## What you should see

The router selects a handler for each case. Compare the routes with the human labels, and inspect both missed reviews and unnecessary reviews. Call counts are estimates and depend on the model’s choices.

## Try it

1. After Step 3, force `routes["Payments"] = ("rule", "forced")` and rerun Step 4. The handler still refuses an automatic retry, and the final `assert` stops the cell — that stop is the evaluation catching the dangerous route. Rerun Step 3 to recover.
2. Make a reply `[]` or give it the route `maybe`. Both fall back to review.
3. Add a timeout budget to Recommendations. When should a fallback replace another attempt?
4. Replace Inventory's unknown idempotency with a documented idempotency key and a reconciliation API. Reconsider its route with a domain expert.

## Takeaway

A router allocates effort; it does not create safety by itself. Deterministic handlers retain the no-blind-payment-retry rule, unknown cases get reviewed, and measured routing errors reveal whether the extra model call was worthwhile.